# 03 — The media-mix model, fitted wrong on purpose

Two fits on the panel with the real Olist baseline. The **misspecified** arm leads,
because it is the situation an analyst is actually in: nobody knows the true
functional form. The **matched** arm is a control, and reporting only it would be
circular.

Priors are pymc-marketing's defaults, unchanged. This matters more than it sounds:
prior choice is where circularity gets into a recovery study without anyone
noticing, and a prior centred near the true ROI would produce excellent recovery
and prove nothing. The consequence is wide intervals, which is a finding rather
than a defect.

In [ ]:
import json
import warnings

import pandas as pd

from athar import paths
from athar.provenance import read_metric

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

METRICS = paths.metrics_dir()
PROCESSED = paths.processed_dir()


def show(frame, caption=""):
    if caption:
        print(caption)
    print(frame.to_string(index=False))
    print()

In [ ]:
mmm = read_metric("mmm", METRICS)
print(mmm["truth_access"])
print()
for name, description in mmm["design"]["specifications"].items():
    print(f"{name:14s} {description}")

## Did the sampler work

A fit that fails its diagnostics is reported as failed rather than folded into a
result. The threshold is a divergence *rate* rather than a count, because a count
tightens as you sample more and is therefore not a rule.

In [ ]:
show(
    pd.DataFrame(
        [
            {
                "specification": name,
                **{
                    k: v
                    for k, v in fit["diagnostics"].items()
                    if k not in ("criteria", "criteria_strict")
                },
            }
            for name, fit in mmm["fits"].items()
        ]
    )
)

## Recovery

Coverage leads. Whether the true value falls inside the interval is the property
the model actually claims, and it is checkable; a point estimate that lands close
on one draw is an anecdote.

In [ ]:
for name, fit in mmm["fits"].items():
    rows = [
        {"channel": c, **{k: round(v, 4) if isinstance(v, float) else v for k, v in entry.items()}}
        for c, entry in fit["average_roi"]["channels"].items()
    ]
    show(
        pd.DataFrame(rows)[
            [
                "channel",
                "true",
                "estimated_mean",
                "hdi_low",
                "hdi_high",
                "covered",
                "relative_error",
            ]
        ],
        f"{name} — average ROI",
    )
    print(
        "  summary:",
        {
            k: round(v, 4) if isinstance(v, float) else v
            for k, v in fit["average_roi"]["summary"].items()
        },
    )
    print()

## Marginal ROI, which is what the budget decision needs

Scored separately, because a model can be respectable on average ROI and useless
on the slope — and the slope is the quantity notebook 08 allocates on.

In [ ]:
for name, fit in mmm["fits"].items():
    summary = fit["marginal_roi"]["summary"]
    print(
        f"{name:14s} coverage {summary['coverage_rate']:.2f}  "
        f"median |rel err| {summary['median_absolute_relative_error']:.2f}  "
        f"mean interval width {summary['mean_interval_width']:.2f}"
    )

## What the model recovered about the transforms

The misspecified arm cannot represent a delayed peak, so whatever the true delay
was has to be absorbed somewhere else — usually into the decay rate and the
coefficient. This is where that shows.

In [ ]:
print(json.dumps(mmm["fits"]["misspecified"]["recovered_parameters"], indent=2)[:1800])